# Module 7 — Binary Trees

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a6-binary-trees/starter/binary_trees.py`.

## 1. TreeNode and the four traversals (Lecture 1)

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

def preorder(node, result=None):
    if result is None: result = []
    if node is not None:
        result.append(node.value)
        preorder(node.left, result)
        preorder(node.right, result)
    return result

def inorder(node, result=None):
    if result is None: result = []
    if node is not None:
        inorder(node.left, result)
        result.append(node.value)
        inorder(node.right, result)
    return result

def postorder(node, result=None):
    if result is None: result = []
    if node is not None:
        postorder(node.left, result)
        postorder(node.right, result)
        result.append(node.value)
    return result

def level_order(root):
    if root is None: return []
    result, q = [], deque([root])
    while q:
        node = q.popleft()
        result.append(node.value)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return result

def tree_height(node):
    if node is None:
        return -1
    return 1 + max(tree_height(node.left), tree_height(node.right))

root = TreeNode(8)
root.left = TreeNode(3); root.right = TreeNode(10)
root.left.left = TreeNode(1); root.left.right = TreeNode(6)
root.right.right = TreeNode(14)

assert preorder(root) == [8, 3, 1, 6, 10, 14]
assert inorder(root) == [1, 3, 6, 8, 10, 14]
assert postorder(root) == [1, 6, 3, 14, 10, 8]
assert level_order(root) == [8, 3, 10, 1, 6, 14]
assert tree_height(root) == 2
print("All four traversals and height check passed, matching Lecture 1 exactly")

All four traversals and height check passed, matching Lecture 1 exactly


## 2. BST search, insert, delete (Lecture 2)

In [2]:
def bst_search(node, target):
    if node is None or node.value == target:
        return node
    if target < node.value:
        return bst_search(node.left, target)
    return bst_search(node.right, target)

def bst_insert(node, value):
    if node is None:
        return TreeNode(value)
    if value < node.value:
        node.left = bst_insert(node.left, value)
    elif value > node.value:
        node.right = bst_insert(node.right, value)
    return node

def bst_delete(node, value):
    if node is None:
        return None
    if value < node.value:
        node.left = bst_delete(node.left, value)
    elif value > node.value:
        node.right = bst_delete(node.right, value)
    else:
        if node.left is None and node.right is None:
            return None
        if node.left is None:
            return node.right
        if node.right is None:
            return node.left
        successor = node.right
        while successor.left is not None:
            successor = successor.left
        node.value = successor.value
        node.right = bst_delete(node.right, successor.value)
    return node

def is_valid_bst(node, low=float('-inf'), high=float('inf')):
    if node is None:
        return True
    if not (low < node.value < high):
        return False
    return is_valid_bst(node.left, low, node.value) and is_valid_bst(node.right, node.value, high)

bst_root = None
for v in [5, 3, 8, 1, 4, 7, 9]:
    bst_root = bst_insert(bst_root, v)
assert inorder(bst_root) == [1, 3, 4, 5, 7, 8, 9]
assert is_valid_bst(bst_root)

bst_root = bst_delete(bst_root, 5)   # two-children case: root deleted
assert is_valid_bst(bst_root)
assert inorder(bst_root) == [1, 3, 4, 7, 8, 9]
print("BST search/insert/delete checks passed, matching Lecture 2's traces")

BST search/insert/delete checks passed, matching Lecture 2's traces


## 3. AVL insertion with rotations (Lecture 3)

In [3]:
class AVLNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None
        self.height = 0

def h(node):
    return node.height if node else -1

def update_height(node):
    node.height = 1 + max(h(node.left), h(node.right))

def balance_factor(node):
    return h(node.left) - h(node.right)

def rotate_left(node):
    new_root = node.right
    node.right = new_root.left
    new_root.left = node
    update_height(node)
    update_height(new_root)
    return new_root

def rotate_right(node):
    new_root = node.left
    node.left = new_root.right
    new_root.right = node
    update_height(node)
    update_height(new_root)
    return new_root

def avl_insert(node, value):
    if node is None:
        return AVLNode(value)
    if value < node.value:
        node.left = avl_insert(node.left, value)
    elif value > node.value:
        node.right = avl_insert(node.right, value)
    else:
        return node
    update_height(node)
    bf = balance_factor(node)
    if bf > 1:
        if balance_factor(node.left) < 0:
            node.left = rotate_left(node.left)     # left-right case
        return rotate_right(node)                    # left-left case
    if bf < -1:
        if balance_factor(node.right) > 0:
            node.right = rotate_right(node.right)   # right-left case
        return rotate_left(node)                      # right-right case
    return node

avl_root = None
for v in [1, 2, 3, 4, 5, 6, 7]:
    avl_root = avl_insert(avl_root, v)
assert avl_root.height == 2   # matches Lecture 3's exercise exactly

plain_root = None
for v in [1, 2, 3, 4, 5, 6, 7]:
    plain_root = bst_insert(plain_root, v)
assert tree_height(plain_root) == 6   # plain BST degenerates, per Lecture 1/2
print("AVL height:", avl_root.height, " Plain BST height:", tree_height(plain_root))
print("AVL rotation checks passed, contrast with plain BST confirmed")

AVL height: 2  Plain BST height: 6
AVL rotation checks passed, contrast with plain BST confirmed


## 4. Range query, floor, ceiling (Lecture 4)

In [4]:
def range_query(node, low, high, result=None):
    if result is None: result = []
    if node is None:
        return result
    if node.value > low:
        range_query(node.left, low, high, result)
    if low <= node.value <= high:
        result.append(node.value)
    if node.value < high:
        range_query(node.right, low, high, result)
    return result

def floor(node, target):
    if node is None:
        return None
    if node.value == target:
        return node.value
    if node.value > target:
        return floor(node.left, target)
    right_floor = floor(node.right, target)
    return right_floor if right_floor is not None else node.value

def ceiling(node, target):
    if node is None:
        return None
    if node.value == target:
        return node.value
    if node.value < target:
        return ceiling(node.right, target)
    left_ceiling = ceiling(node.left, target)
    return left_ceiling if left_ceiling is not None else node.value

assert range_query(root, 3, 9) == [3, 6, 8]     # matches Lecture 4's exact trace
assert floor(root, 7) == 6                        # matches Lecture 4's exact trace
assert ceiling(root, 7) == 8
print("Range query, floor, and ceiling checks passed, matching Lecture 4's traces")

Range query, floor, and ceiling checks passed, matching Lecture 4's traces


## 5. Treesort (Lecture 4)

In [5]:
def treesort(arr):
    t_root = None
    for value in arr:
        t_root = bst_insert(t_root, value)
    return inorder(t_root)

assert treesort([5, 2, 8, 1, 9]) == [1, 2, 5, 8, 9]
print("Treesort check passed, matching Lecture 4's trace")

Treesort check passed, matching Lecture 4's trace
